In [ ]:
%load_ext watermark


In [ ]:
from IPython.display import display, HTML
from matplotlib import pyplot as plt
import pandas as pd
import polars as pl
import seaborn as sns
from slugify import slugify
from teeplot import teeplot as tp

from pylib._percentilestatcat_plot import (
    percentilestatcat_plot,
)
from pylib._seed_global_rngs import seed_global_rngs


In [ ]:
%watermark -diwmuv -iv


In [ ]:
teeplot_subdir = "2025-05-17-vanilla-compscreen"
teeplot_subdir


In [ ]:
seed_global_rngs(1)


## Get Data


In [ ]:
url = "https://osf.io/aysxt/download"


In [ ]:
df = pl.scan_parquet(
    url,
    low_memory=True,
    retries=5,
)
schema = df.collect_schema()


In [ ]:
fil = (
    df.filter(
        pl.col("trt_hsurf_bits").eq(0),
    ).filter(
        pl.col("replicate_uuid").eq(
            pl.col("replicate_uuid").first().over("trt_name"),
        )
    )
    .select(
        pl.exclude([k for k, v in schema.items() if v == pl.String]),
    )
    .collect()
)


## Plot Data


In [ ]:
for (trt_name,), group in fil.group_by("trt_name"):
    display(HTML(f"<h1>{trt_name}</h1>"))

    dfx = group.to_pandas()
    dfx_ = group.to_pandas()
    dfx_["is_focal_defmut"] = "null"
    data = pd.concat([dfx, dfx_], ignore_index=True)

    for y in (
        "defmut_norm_all-num_leaves",
        "defmut_norm_ot_bin:week-num_leaves",
        "defmut_norm_ot_bin:fortnight-num_leaves",
        "defmut_norm_ot_bin:month-num_leaves",
        "defmut_norm_ot_bin:quarter-num_leaves",
        "defmut_norm_ot_bin:year-num_leaves",
        "defmut_norm_match:variant_flavor-num_leaves",
        "defmut_norm_all-clade_duration",
        "defmut_norm_ot_bin:week-clade_duration",
        "defmut_norm_ot_bin:fortnight-clade_duration",
        "defmut_norm_ot_bin:month-clade_duration",
        "defmut_norm_ot_bin:quarter-clade_duration",
        "defmut_norm_ot_bin:year-clade_duration",
        "defmut_norm_match:variant_flavor-clade_duration",
    ):
        display(HTML(f"<h2>{trt_name} {y}</h2>"))
        plt.close("all")  # release resources
        with tp.teed(
            percentilestatcat_plot,
            data=data,
            x="is_focal_defmut",
            y=y,
            hue="is_focal_defmut",
            teeplot_outattrs={
                "trt_name": slugify(trt_name),
            },
            teeplot_subdir=teeplot_subdir,
        ):
            pass


In [ ]:
for (trt_name,), group in fil.group_by("trt_name"):
    display(HTML(f"<h1>{trt_name}</h1>"))

    for norm in (
        "all",
        "match:variant_flavor",
        "ot_bin:week",
        "ot_bin:fortnight",
        "ot_bin:month",
        "ot_bin:quarter",
        "ot_bin:year",
    ):

        for var in "num_leaves", "clade_duration":
            display(HTML(f"<h2>{trt_name} {norm} {var}</h2>"))

            plt.close("all")  # release resources
            with tp.teed(
                sns.scatterplot,
                data=group,
                y=f"defmut_norm_{norm}-{var}",
                x=var,
                hue={
                    "all": None,
                    "ot_bin": "origin_time",
                    "match": "variant_flavor",
                }[norm.split(":")[0]],
                teeplot_outattrs={
                    "trt_name": slugify(trt_name),
                },
                teeplot_subdir=teeplot_subdir,
            ) as teed:
                teed.set_xscale(
                    {
                        "num_leaves": "log",
                        "clade_duration": "symlog",
                    }[var],
                )
                teed.set_ylim(-5, 105)
                teed.set_xlim(
                    {
                        "num_leaves": None,
                        "clade_duration": -0.5,
                    }[var],
                    None,
                )
                teed.spines[['right', 'top']].set_visible(False)
